# HydroSeason Quickstart

Install with `pip install hydroseason`, then run this notebook from anywhere.

Minimal notebook workflow:
1. Load monthly data
2. Run hydrological delineation
3. Inspect the summary card (regime, SI, key stats)
4. Explore interactive season plots
5. Export a self-contained HTML report

In [7]:
import importlib
import hydroseason.plot as hp
importlib.reload(hp)

<module 'hydroseason.plot' from 'D:\\RLH\\5.6\\repos\\hydroseason\\hydroseason\\plot.py'>

In [8]:
from pathlib import Path

from hydroseason import read_rainfall

data_path = Path("../data/monthly_rainfall.csv")
df = read_rainfall(data_path, source="csv")
df.head()

,Date,Year,Month,Rainfall_mm
0,1986-12-01,1986,12,26.5
1,1987-01-01,1987,1,214.5
2,1987-02-01,1987,2,220.5
3,1987-03-01,1987,3,149.5
4,1987-04-01,1987,4,2.5


In [9]:
from hydroseason import classify_rainfall
from hydroseason.report import display_summary

artifacts = classify_rainfall(df)
result = artifacts.result

result.head()

,Date,Year,Month,Rainfall_mm,Imputed,Hydro_Year_fixed,Smoothed,Significant,SeasonType,SeasonShift,...,dry_month_count,wet_month_count,Dry_season_rain_count,Rain_dry_season_mm,Rain_wet_season_mm,Dry_month_count,Rain_Smoothed,Annual_SPI,Year_Class_SPI,Drought_Category
0,1986-12-01,1986,12,26.5,False,1987,120.500000,True,Wet,True,...,8,4,4,56.0,611.0,8,120.500000,-0.341,Regular,Prolonged
1,1987-01-01,1987,1,214.5,False,1987,153.833333,True,Wet,False,...,8,4,4,56.0,611.0,8,153.833333,-0.341,Regular,Prolonged
2,1987-02-01,1987,2,220.5,False,1987,194.833333,True,Wet,False,...,8,4,4,56.0,611.0,8,194.833333,-0.341,Regular,Prolonged
3,1987-03-01,1987,3,149.5,False,1987,124.166667,True,Wet,False,...,8,4,4,56.0,611.0,8,124.166667,-0.341,Regular,Prolonged
4,1987-04-01,1987,4,2.5,False,1987,60.333333,True,Dry,True,...,8,4,4,56.0,611.0,8,60.333333,-0.341,Regular,Prolonged


In [10]:
display_summary(artifacts)

### What `classify_rainfall` returns

`artifacts` is a `PipelineArtifacts` bundle with five fields:

- **`result`** — your input rows plus `SeasonType`, `Hydro_Year`, and metric columns (this is the main output).
- **`fixed_monthly`** — the 12-row monthly climatology and baseline Wet/Dry label per calendar month.
- **`wet_boundaries`** — per-hydro-year wet-season start/end boundaries (`None` for non-seasonal regimes).
- **`seasonality`** — STL strength, Walsh-Lawler SI, and the detected regime.
- **`diagnostics`** — a record of every algorithm decision (also written to the `.HydroSeason.json` sidecar).

If validation fails (for example a missing `Rainfall_mm` column or a data gap longer than `max_consecutive_imputation_gap` months), the call raises with a message describing the problem.

In [11]:
from hydroseason.plot import plot_imputation_overview, show

print(f"Data confidence: {artifacts.diagnostics.data_confidence}")
print(f"Imputed months: {artifacts.diagnostics.n_imputed}")
show(plot_imputation_overview(result))

Data confidence: high
Imputed months: 0


In [13]:
from hydroseason.plot import plot_season_timeline, show

# Interactive bar chart — coloured by SeasonType, hydro-year boundaries, range slider
# show() enables scroll-to-zoom and responsive sizing
show(plot_season_timeline(result))

In [14]:
from hydroseason.plot import plot_agg_monthly_rainfall, show

# Mean monthly rainfall coloured by baseline season assignment, with std error bars
show(plot_agg_monthly_rainfall(result, artifacts.fixed_monthly))

In [15]:
from hydroseason.plot import plot_annual_metrics, show

# Stacked wet/dry totals per hydrological year + wet month count
show(plot_annual_metrics(result))

In [16]:
from hydroseason.plot import plot_dashboard, show

# Composite dashboard: timeline + monthly rainfall + annual totals in one figure
show(plot_dashboard(artifacts))

In [17]:
from hydroseason.report import generate_html_report

# Export a self-contained HTML report (open in any browser — no Python needed)
report_path = generate_html_report(artifacts, "hydroseason_report.html")
print(f"Report written to: {report_path.resolve()}")

Report written to: D:\RLH\5.6\repos\hydroseason\notebooks\hydroseason_report.html


In [18]:
from hydroseason.report import export_bundle

# Export full bundle: offline HTML report + CSV + JSON.
# Add export_png=True when static PNG files are needed and Kaleido/Chrome are available.
bundle_path = export_bundle(artifacts, "hydroseason_export")
print(f"Bundle written to: {bundle_path}")
print("Contents:")
for p in sorted(bundle_path.rglob("*")):
    print(f"  {p.relative_to(bundle_path)}")

Bundle written to: D:\RLH\5.6\repos\hydroseason\notebooks\hydroseason_export
Contents:
  data
  data\diagnostics.json
  data\metrics_annual.csv
  data\results_monthly.csv
  figures
  report.html
